In [ ]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np
from datetime import datetime

# To suppress scientific notations
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# To tune model, get different metric scores, and split data
from sklearn import metrics
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# To be used for data scaling and one hot encoding
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder

# To impute missing values
from sklearn.impute import SimpleImputer


# To oversample and undersample data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To do hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# To define maximum number of columns to be displayed in a dataframe
pd.set_option("display.max_columns", None)

# To supress scientific notations for a dataframe
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# To help with model building
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    BaggingClassifier,
)
from xgboost import XGBClassifier

# To supress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Traveldata_train = pd.read_csv('Traveldata_train.csv')
Traveldata_test = pd.read_csv('Traveldata_test.csv')

Surveydata_train = pd.read_csv('Surveydata_train.csv')
Surveydata_test = pd.read_csv('Surveydata_test.csv')


In [ ]:
Data = pd.merge(Traveldata_train, Surveydata_train, on="ID", how="inner")


In [ ]:
Test_data = pd.merge(Traveldata_test, Surveydata_test, on="ID", how="inner")


In [ ]:
# copying data to another variable to avoid any changes to original data
data = Data.copy()

In [ ]:
Tdata = Test_data.copy()

In [ ]:
# checking for null values
data.isnull().sum()

In [ ]:
# checking for null values
Tdata.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.head()

In [ ]:
data.Overall_Experience.nunique()

In [ ]:
# Check for missing values after dropping the column
missing_values = (data.isnull().sum() / data.shape[0]) * 100

In [ ]:
# Check for missing values after dropping the column
Tmissing_values = (Tdata.isnull().sum() / Tdata.shape[0]) * 100

In [ ]:
# checking for null values
Tmissing_values

In [ ]:
Tdata

In [ ]:
# Define the mapping for ordinal columns
ordinal_mapping = {
    "Extremely Poor": 1,
    "Poor": 2,
    "Needs Improvement": 3,
    "Acceptable": 4,
    "Good": 5,
    "Excellent": 6
}

# List of columns to encode
ordinal_columns = [
    "Arrival_Time_Convenient","Seat_Comfort", "Catering", "Onboard_Wifi_Service",
    "Onboard_Entertainment", "Online_Support", "Ease_of_Online_Booking",
    "Onboard_Service", "Legroom", "Baggage_Handling", "CheckIn_Service",
    "Cleanliness", "Online_Boarding"
]

# Apply the mapping to encode ordinal columns
for column in ordinal_columns:
    data[column] = data[column].map(ordinal_mapping)


In [ ]:
# Define the mapping for Platform_Location
platform_location_mapping = {
    "Very Inconvenient": 1,
    "Inconvenient": 2,
    "Needs Improvement": 3,
    "Manageable": 4,
    "Convenient": 5,
    "Very Convenient": 6
}

# Apply the mapping to encode Platform_Location
data["Platform_Location"] = data["Platform_Location"].map(platform_location_mapping)

In [ ]:
data.head()

In [ ]:
# Apply the mapping to encode ordinal columns
for column in ordinal_columns:
    Tdata[column] = Tdata[column].map(ordinal_mapping)

In [ ]:
# Apply the mapping to encode Platform_Location
Tdata["Platform_Location"] = Tdata["Platform_Location"].map(platform_location_mapping)

In [ ]:
numerical_cols = data.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()

cols_list = numerical_cols + categorical_cols

cols_list

In [ ]:
Tnumerical_cols = Tdata.select_dtypes(include=["number"]).columns.tolist()
Tcategorical_cols = Tdata.select_dtypes(include=["object"]).columns.tolist()

Tcols_list = Tnumerical_cols + Tcategorical_cols

Tcols_list

In [ ]:
data = data.dropna(axis=0)

In [ ]:
# Impute numerical columns with median
  # Replace with relevant numerical column names
Tmedian_imputer = SimpleImputer(strategy="median")
Tdata[Tnumerical_cols] = Tmedian_imputer.fit_transform(Tdata[Tnumerical_cols])

In [ ]:
# Impute categorical columns with mode
Tmode_imputer = SimpleImputer(strategy="most_frequent")
Tdata[Tcategorical_cols] = Tmode_imputer.fit_transform(Tdata[Tcategorical_cols])

In [ ]:
data = data.drop(["ID"], axis=1)

In [ ]:
# Check for missing values after dropping the column
missing_values = (data.isnull().sum() / data.shape[0]) * 100

# checking for null values
missing_values

In [ ]:
# Check for missing values after dropping the column
Tmissing_values = (Tdata.isnull().sum() / Tdata.shape[0]) * 100

# checking for null values
Tmissing_values

In [ ]:
# defining X and y variables
X = data.drop(["Overall_Experience"], axis=1)
y = data["Overall_Experience"]

print(X.head())
print(y.head())

In [ ]:
# creating dummy variables
X = pd.get_dummies(
    X,
    columns=X.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
X = X.astype(float)
X.head()

In [ ]:
# outlier detection using boxplot
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
# dropping release_year as it is a temporal variable

plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

In [ ]:
# creating dummy variables 
X_Test = pd.get_dummies(
    Tdata,
    columns=Tdata.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
X_Test = X_Test.astype(float)
X_Test.head()

In [ ]:


# then we split the temporary set into train and validation

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)
print(X_train.shape, X_val.shape, X_Test.shape)

# Model Building

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

## Initial Model Building

### Model Building - Original Data

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

print("\nTraining Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores = recall_score(y_train, model.predict(X_train))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores_val = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores_val))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train, y_train)
    scores_train = recall_score(y_train, model.predict(X_train))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference1 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference1))

- Adaboost has the best performance followed by GBM model as per the validation performance

### Model Building - Oversampled Data

In [ ]:
print("Before Oversampling, counts of label 'Yes': {}".format(sum(y_train == 1)))
print("Before Oversampling, counts of label 'No': {} \n".format(sum(y_train == 0)))

sm = SMOTE(
    sampling_strategy=1, k_neighbors=5, random_state=1
)  # Synthetic Minority Over Sampling Technique
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


print("After Oversampling, counts of label 'Yes': {}".format(sum(y_train_over == 1)))
print("After Oversampling, counts of label 'No': {} \n".format(sum(y_train_over == 0)))


print("After Oversampling, the shape of train_X: {}".format(X_train_over.shape))
print("After Oversampling, the shape of train_y: {} \n".format(y_train_over.shape))

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

print("\n" "Training Performance:" "\n")
for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores = recall_score(y_train_over, model.predict(X_train_over))
    print("{}: {}".format(name, scores))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores_train = recall_score(y_train_over, model.predict(X_train_over))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference2 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference2))

- Adaboost has the best performance on validation followed by GBM

### Model Building - Undersampled Data

In [ ]:
rus = RandomUnderSampler(random_state=1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)

In [ ]:
print("Before Under Sampling, counts of label 'Yes': {}".format(sum(y_train == 1)))
print("Before Under Sampling, counts of label 'No': {} \n".format(sum(y_train == 0)))

print("After Under Sampling, counts of label 'Yes': {}".format(sum(y_train_un == 1)))
print("After Under Sampling, counts of label 'No': {} \n".format(sum(y_train_un == 0)))

print("After Under Sampling, the shape of train_X: {}".format(X_train_un.shape))
print("After Under Sampling, the shape of train_y: {} \n".format(y_train_un.shape))

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))


print("\n" "Training Performance:" "\n")
for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores = recall_score(y_train_un, model.predict(X_train_un))
    print("{}: {}".format(name, scores))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores_train = recall_score(y_train_un, model.predict(X_train_un))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference3 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference3))

- GBM has the best performance followed by AdaBoost model as per the validation performance

## Hyperparameter Tuning

### Tuning AdaBoostClassifier model with Original data

In [ ]:
%%time

# defining model
Model = AdaBoostClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(10, 40, 10),
    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],
    "estimator": [
        DecisionTreeClassifier(max_depth=1, random_state=1),
        DecisionTreeClassifier(max_depth=2, random_state=1),
        DecisionTreeClassifier(max_depth=3, random_state=1),
    ],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_jobs = -1, n_iter=50, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_adb = AdaBoostClassifier(
    random_state=1,
    n_estimators=30,
    learning_rate=1,
    estimator=DecisionTreeClassifier(max_depth=3, random_state=1),
)
tuned_adb.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
adb_train = model_performance_classification_sklearn(tuned_adb, X_train, y_train)
adb_train

In [ ]:
# Checking model's performance on validation set
adb_val = model_performance_classification_sklearn(tuned_adb, X_val, y_val)
adb_val

### Tuning  Gradient Boosting model with Original Data

In [ ]:
%%time

#Creating pipeline
Model = GradientBoostingClassifier(random_state=1)

#Parameter grid to pass in RandomSearchCV
param_grid = {
    "init": [AdaBoostClassifier(random_state=1),DecisionTreeClassifier(random_state=1)],
    "n_estimators": np.arange(125,175,25),
    "learning_rate": [0.01, 0.2, 0.05, 1],
    "subsample":[0.8,0.9,1],
    "max_features":[0.5,0.7,1],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=50, scoring=scorer, cv=5, random_state=1, n_jobs = -1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_gbm1 = GradientBoostingClassifier(
    random_state=1,
    subsample=1,
    n_estimators=150,
    max_features=0.7,
    learning_rate=1,
    init=AdaBoostClassifier(random_state=1),
)
tuned_gbm1.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
gbm1_train = model_performance_classification_sklearn(
    tuned_gbm1, X_train, y_train)
gbm1_train

In [ ]:
# Checking model's performance on validation set
gbm1_val = model_performance_classification_sklearn(tuned_gbm1, X_val, y_val)
gbm1_val

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        adb_train.T,
        gbm1_train.T,
            ],
    axis=1,
)
models_train_comp_df.columns = [
    "AdaBoost boosting trained with Original data",
    "Gradient boosting trained with Original data",
    ]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# Validation performance comparison

models_train_comp_df = pd.concat(
    [adb_val.T, gbm1_val.T ], axis=1,
)
models_train_comp_df.columns = [
    "AdaBoost boosting trained with Original data",
    "Gradient boosting trained with Original data",
    ]
print("Validation performance comparison:")
models_train_comp_df

In [ ]:
%%time

#Creating pipeline
Model = RandomForestClassifier(random_state=1)

#Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(150,200,250),
    "max_features":[0.5,0.7,1],
    "max_depth": [3, 5, None],  # Add max depth if needed
    "min_samples_split": [2, 5, 10]  # Minimum samples for splitting nodes
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=50, scoring=scorer, cv=5, random_state=1, n_jobs = -1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_Random_Forest = RandomForestClassifier(
    random_state=1,
    n_estimators=250,  # Number of trees in the forest
    max_depth=None       # Maximum depth of each tree
)
tuned_Random_Forest.fit(X_train, y_train)

### Feature Importance

In [ ]:
feature_names = X_train.columns
importances = tuned_Random_Forest.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

In [ ]:
print("Trained Features:", tuned_Random_Forest.feature_names_in_)

In [ ]:
# Reorder test data columns to match trained feature names
test_data = X_Test[tuned_Random_Forest.feature_names_in_]

In [ ]:
# Predict the target variable using the trained model

test_data["Overall_Experience"] = tuned_Random_Forest.predict(test_data)

In [ ]:
id_column = X_Test["ID"]

In [ ]:
test_data["ID"] = id_column

In [ ]:
test_data.to_csv('tuned_Random_Forest.csv', index=False)